In [13]:
import numpy as np
import pandas as pd
import os
import glob
import re
from sklearn.preprocessing import StandardScaler

In [14]:
def _compute_window_band_powers(vals, fs, window_size, bands, center=True, pad=True, window_type='hann'):
    n = len(vals)
    if n == 0:
        return np.zeros((0, len(bands)))

    # 窓関数
    if window_type in ['hann', 'hanning']:
        win = np.hanning(window_size)
    else:
        win = np.ones(window_size)

    # 周波数軸（必ず window_size を使う）
    freqs = np.fft.rfftfreq(window_size, d=1.0/fs)

    band_mat = np.zeros((n, len(bands)), dtype=float)

    for i in range(n):
        # ウィンドウ位置決定
        if center:
            start = i - window_size // 2
        else:
            start = i
        end = start + window_size

        # 切り出し＋パディング
        if start < 0 or end > n:
            if pad:
                s_idx = max(start, 0)
                e_idx = min(end, n)
                seg = vals[s_idx:e_idx]
                left = s_idx - start
                right = end - e_idx
                seg_padded = np.pad(seg, (max(0, left), max(0, right)), mode='reflect')

                if len(seg_padded) < window_size:
                    seg_padded = np.pad(seg_padded, (0, window_size - len(seg_padded)), mode='edge')

                windowed = seg_padded[:window_size]
            else:
                s_idx = max(start, 0)
                e_idx = min(end, n)
                seg = vals[s_idx:e_idx]
                if len(seg) < window_size:
                    windowed = np.pad(seg, (0, window_size - len(seg)), mode='constant')
                else:
                    windowed = seg[:window_size]
        else:
            windowed = vals[start:end]

        # DC成分除去
        windowed = windowed - np.mean(windowed)

        # 窓関数適用
        windowed = windowed * win

        # --- FFT ---
        fft_vals = np.fft.rfft(windowed)

        # 振幅（正規化：2/N）
        amp = (2.0 / window_size) * np.abs(fft_vals)

        # パワー
        power = amp ** 2

        # --- バンドごとに合計 ---
        for b_i, (f_lo, f_hi) in enumerate(bands):
            mask = (freqs >= f_lo) & (freqs < f_hi)
            band_mat[i, b_i] = float(np.sum(power[mask])) if np.any(mask) else 0.0

    return band_mat


# 加速度前処理（特徴量作成）
def preprocess_acc_sensor_minimal(df_raw, sensor_suffix="acc1",
                                  fs=100.0, window_size=1024,
                                  bands=[(0.2, 0.7), (0.7, 1.5), (1.5, 3.0), (3.0, 6.0), (6.0, 12.0), (12.0, 25.0)],
                                  center=True, pad=True,
                                  window_stat=100):
    df = df_raw.copy()

    # ---- 時刻処理 ----
    df["ArriveTime"] = pd.to_datetime(df.get("ArriveTime"), errors="coerce")
    df["ArriveTime"] = df["ArriveTime"].ffill()
    df["time"] = df["ArriveTime"]

    # ---- 加速度を float へ ----
    ax = pd.to_numeric(df.get("AccelerationX", 0), errors="coerce").fillna(0.0).astype(float)
    ay = pd.to_numeric(df.get("AccelerationY", 0), errors="coerce").fillna(0.0).astype(float)
    az = pd.to_numeric(df.get("AccelerationZ", 0), errors="coerce").fillna(0.0).astype(float)

    out = pd.DataFrame(index=df.index)

    # ---- 基本値 ----
    out[f"time_{sensor_suffix}"] = df["time"].values
    out[f"acc_x_{sensor_suffix}"] = ax
    out[f"acc_y_{sensor_suffix}"] = ay
    out[f"acc_z_{sensor_suffix}"] = az

    # ---- magnitude ----
    mag = np.sqrt(ax**2 + ay**2 + az**2)
    out[f"acc_mag_{sensor_suffix}"] = mag

    # ---- 差分 ----
    out[f"acc_mag_diff_{sensor_suffix}"] = np.r_[0.0, np.diff(mag)]
    out[f"acc_x_diff_{sensor_suffix}"] = np.r_[0.0, np.diff(ax)]
    out[f"acc_y_diff_{sensor_suffix}"] = np.r_[0.0, np.diff(ay)]
    out[f"acc_z_diff_{sensor_suffix}"] = np.r_[0.0, np.diff(az)]

    # ============================================
    #   ★★★ 追加した特徴量 ★★★
    # ============================================

    # --- mean_x / mean_y / mean_z（移動平均） ---
    out[f"mean_x_{sensor_suffix}"] = ax.rolling(window_stat, min_periods=1).mean()
    out[f"mean_y_{sensor_suffix}"] = ay.rolling(window_stat, min_periods=1).mean()
    out[f"mean_z_{sensor_suffix}"] = az.rolling(window_stat, min_periods=1).mean()

    # --- std_x / std_y / std_z（移動 std） ---
    out[f"std_x_{sensor_suffix}"] = ax.rolling(window_stat, min_periods=1).std().fillna(0)
    out[f"std_y_{sensor_suffix}"] = ay.rolling(window_stat, min_periods=1).std().fillna(0)
    out[f"std_z_{sensor_suffix}"] = az.rolling(window_stat, min_periods=1).std().fillna(0)

    # --- mean_mag / std_mag ---
    out[f"mean_mag_{sensor_suffix}"] = mag.rolling(window_stat, min_periods=1).mean()
    out[f"std_mag_{sensor_suffix}"]  = mag.rolling(window_stat, min_periods=1).std().fillna(0)

    # --- SMA（瞬間値）---
    out[f"sma_{sensor_suffix}"] = (np.abs(ax) + np.abs(ay) + np.abs(az)) / 3.0

    # --- int_abs（mag の積分値）---
    dt = 1.0 / fs
    out[f"int_abs_{sensor_suffix}"] = np.cumsum(np.abs(mag)) * dt

    # ============================================
    #   ここまで追加特徴量
    # ============================================

    # ---- バンドパワー ----
    band_x = _compute_window_band_powers(ax.values, fs, window_size, bands, center, pad)
    band_y = _compute_window_band_powers(ay.values, fs, window_size, bands, center, pad)
    band_z = _compute_window_band_powers(az.values, fs, window_size, bands, center, pad)
    band_m = _compute_window_band_powers(mag, fs, window_size, bands, center, pad)

    for b_i, (f_lo, f_hi) in enumerate(bands):
        bi = b_i + 1
        out[f"acc_x_band{bi}_power_win_{f_lo}-{f_hi}Hz_{sensor_suffix}"] = band_x[:, b_i]
        out[f"acc_y_band{bi}_power_win_{f_lo}-{f_hi}Hz_{sensor_suffix}"] = band_y[:, b_i]
        out[f"acc_z_band{bi}_power_win_{f_lo}-{f_hi}Hz_{sensor_suffix}"] = band_z[:, b_i]
        out[f"acc_mag_band{bi}_power_win_{f_lo}-{f_hi}Hz_{sensor_suffix}"] = band_m[:, b_i]

    return out


In [15]:
def load_acc_only(
        keyword,
        root_folder=r"../../../../../デスクトップ/実験",
        fs=100.0,
        window_size=1024,
        bands=[(0.2, 0.7), (0.7, 1.5), (1.5, 3.0), (3.0, 6.0), (6.0, 12.0), (12.0, 25.0)],
        center=True,
        pad=True,
        interpolate_missing=True,
        save_path=None):

    keyword = str(keyword)

    # 加速度フォルダ
    acc_path = os.path.join(root_folder, "data_acc")

    # ファイル検索：keyword + 10D739C を含むものだけ
    acc_all = glob.glob(os.path.join(acc_path, "*.csv"))
    acc_files = [
        f for f in acc_all
        if (keyword.lower() in os.path.basename(f).lower()) and
           ("10D739C".lower() in os.path.basename(f).lower())
    ]

    if len(acc_files) == 0:
        raise FileNotFoundError(f"keyword='{keyword}', '10D739C' を含む加速度ファイルが見つかりません")

    acc_file = acc_files[0]
    print("---- ACC 選択 ----")
    print("acc file:", os.path.basename(acc_file))

    # --- 加速度ロード ---
    df = pd.read_csv(acc_file)
    df.columns = df.columns.str.strip()

    # 時刻処理（あなたのロジックを保持）
    df["ArriveTime"] = pd.to_datetime(df.get("ArriveTime"), errors="coerce").ffill()
    df["time"] = df["ArriveTime"]

    # 数値変換（旧ロジック）
    df["AccelerationX"] = pd.to_numeric(df.get("AccelerationX"), errors="coerce").fillna(0.0)
    df["AccelerationY"] = pd.to_numeric(df.get("AccelerationY"), errors="coerce").fillna(0.0)
    df["AccelerationZ"] = pd.to_numeric(df.get("AccelerationZ"), errors="coerce").fillna(0.0)

    # --- 特徴量計算 ---
    acc_feat = preprocess_acc_sensor_minimal(
        df, "acc", fs, window_size, bands, center, pad
    )

    # 欠損補完
    if interpolate_missing:
        acc_feat = acc_feat.interpolate("linear").ffill().bfill()
        
    # ★ datetime列を全て除去（標準化に備える）
    acc_feat = acc_feat.select_dtypes(include=["float", "int"])

    # 最初の300データを落とす
    acc_feat = acc_feat.iloc[450:, :].reset_index(drop=True)
    
    # 標準化
    sc = StandardScaler()
    acc_feat = pd.DataFrame(sc.fit_transform(acc_feat), columns=acc_feat.columns)

    # 保存
    if save_path is not None:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        acc_feat.to_csv(save_path, index=False)
        print("✔ 保存:", save_path)

    return acc_feat


In [ ]:
keywords = ['kannno1','kannno2', 'sato1', 'sato2', 'nishi2', 'morita1','morita2','mori1','mori2']
for kw in keywords:
    save_path = f"data_acc/{kw}_accdata.csv"
    df = load_acc_only(kw, save_path=save_path)

---- ACC 選択 ----
acc file: kannno1_AppTag_10D739C_ADXL34x_20251116.csv
✔ 保存: data_acc/kannno1_accdata.csv
---- ACC 選択 ----
acc file: kannno2_AppTag_10D739C_ADXL34x_20251116.csv
✔ 保存: data_acc/kannno2_accdata.csv
---- ACC 選択 ----
acc file: sato1_AppTag_10D739C_ADXL34x_20251116.csv
✔ 保存: data_acc/sato1_accdata.csv
---- ACC 選択 ----
acc file: sato2_AppTag_10D739C_ADXL34x_20251116.csv
✔ 保存: data_acc/sato2_accdata.csv
---- ACC 選択 ----
acc file: nishi2_AppTag_10D739C_ADXL34x_20251122.csv
✔ 保存: data_acc/nishi2_accdata.csv
---- ACC 選択 ----
acc file: morita1_AppTag_10D739C_ADXL34x_20251116.csv
✔ 保存: data_acc/morita1_accdata.csv
---- ACC 選択 ----
acc file: morita2_AppTag_10D739C_ADXL34x_20251116.csv
✔ 保存: data_acc/morita2_accdata.csv
---- ACC 選択 ----
acc file: mori1_AppTag_10D739C_ADXL34x_20251123.csv
✔ 保存: data_acc/mori1_accdata.csv
---- ACC 選択 ----
acc file: mori2_AppTag_10D739C_ADXL34x_20251123.csv
✔ 保存: data_acc/mori2_accdata.csv
